# Método del gradiente conjugado

El método del Gradiente Conjugado (CG) está diseñado para resolver sistemas de ecuaciones lineales de la forma

\begin{equation}
 Ax = b,
\end{equation}
donde $A$ es una matriz simétrica y definida positiva. Esto es equivalente a encontrar el mínimo de la función cuadrática

\begin{equation}
 f(x) = \frac{1}{2} x^T A x - b^T x.
\end{equation}

En contraste con el método de gradiente descendente, el CG elige direcciones de búsqueda $p_{k}$ que son $A$-ortogonales (conjugadas). Esto garantiza convergencia exacta en a lo mucho $n$ pasos para un sistema de $n \times n$.

## Algoritmo de CG

1. Dado un vector inicial $x_{0}$, calculamos el residuo inicial $r_{0} = b - A x_{0}$.
2. La primera dirección de búsqueda es el residuo: $p_{0} = r_{0}$.
3. En cada iteración $k$, calculamos el tamaño del paso $\alpha_{k}$ que minimiza $f(x)$ a lo largo de $p_{k}$

\begin{equation}
 \alpha_{k} = \frac{r_{k}^T r_{k}}{p_{k}^T A p_{k}}.
\end{equation}

4. Actualizamos la solución y el nuevo residuo

\begin{equation}
 x_{k+1} = x_{k} + \alpha_{k} p_{k},
\end{equation}

\begin{equation}
 r_{k+1} = r_{k} - \alpha_{k} A p_{k}.
\end{equation}

5. Calculamos el factor de conjugación $\beta_k$ para asegurar que la nueva dirección sea ortogonal a las anteriores respecto a $A$

\begin{equation}
 \beta_{k} = \frac{r_{k+1}^T r_{k+1}}{r_{k}^T r_{k}}.
\end{equation}

6. Actualizamos la dirección de búsqueda

\begin{equation}
 p_{k+1} = r_{k+1} + \beta_{k} p_{k}.
\end{equation}

In [1]:
import numpy as np
import matplotlib.pyplot as plt

# Suprimir advertencias menores de integración durante los bucles de optimización
import warnings
warnings.filterwarnings("ignore")

# Configurar el estilo de graficación para una mejor visualización
plt.style.use('ggplot')

In [2]:
def conjugate_gradient(A, b, x0=None, tol=1e-8, max_iter=1000):
    """Resuelve Ax = b usando Gradiente Conjugado.
    
    Parámetros:
    -----------
    A : array-like o matriz dispersa (shape: n x n)
        Matriz simétrica y definida positiva.
    b : array-like (shape: n,)
        Lado derecho del sistema.
    x0 : array-like, opcional
        Estimación inicial. Si es None, se inicializa en ceros.
    tol : float
        Tolerancia para la convergencia basada en la norma del residuo.
    max_iter : int
        Número máximo de iteraciones permitidas.
        
    Retorna:
    --------
    x : np.ndarray
        Solución aproximada del sistema Ax = b.
    """
    b = np.asarray(b, dtype=float)
    x = np.array(x0, dtype=float) if x0 is not None else np.zeros_like(b)
    
    r = b - A @ x
    p = r.copy()
    rsold = r @ r  # Producto interno seguro para vectores 1D
    
    for i in range(max_iter):
        Ap = A @ p
        denom = p @ Ap
        
        # Validación para evitar división por cero si A no es definida positiva
        if denom <= 0:
            print(f"En la iteración {i+1}, p.T @ Ap es menor o igual a cero ({denom}). La matriz A podría no ser simétrica definida positiva.")
            break
            
        alpha = rsold / denom
        
        x = x + alpha * p
        r = r - alpha * Ap
        
        rsnew = r @ r
        if np.sqrt(rsnew) < tol:
            print(f"CG convergió exitosamente en {i+1} iteraciones.")
            return x
            
        p = r + (rsnew / rsold) * p
        rsold = rsnew
        
    print(f"CG alcanzó el número máximo de iteraciones ({max_iter}) sin converger a la tolerancia tol={tol}.")
    return x

In [3]:
# Matriz de coeficientes del sistema de ecuaciones lineales.
A = np.array([
    [9, 3, 3, 0, 1],
    [4, 4, 5, 5, 8],
    [3, 5, 8, 3, 6],
    [8, 4, 4, 6, 1],
    [5, 6, 8, 2, 1]
], dtype=float)

# Vector de términos independientes (entradas) del sistema.
B = np.array([4, 0, 6, 8, 5], dtype=float)

x_sol = conjugate_gradient(A, B)

print("Solución CG:", x_sol)

En la iteración 2, p.T @ Ap es menor o igual a cero (-81.2428852550057). La matriz A podría no ser simétrica definida positiva.
CG alcanzó el número máximo de iteraciones (1000) sin converger a la tolerancia tol=1e-08.
Solución CG: [0.25212338 0.         0.37818507 0.50424676 0.31515422]
